# 📊 01 — Dataset Exploration
**Tujuan:** Eksplorasi dataset Kaggle RFT (River Floating Trash) sebelum training.

Notebook ini mencakup:
1. Verifikasi dataset Kaggle RFT (8 kelas)
2. Analisis distribusi kelas
3. Analisis ukuran objek (untuk menentukan threshold 'objek kecil')
4. Visualisasi contoh anotasi
5. Ringkasan struktur YOLO dataset


In [ ]:
import sys
sys.path.insert(0, '..')  # Add project root to path

import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from collections import Counter
import cv2
from PIL import Image

# Project config
import yaml
with open('../config/config.yaml') as f:
    cfg = yaml.safe_load(f)

print('✅ Config loaded')
print(f'Project: {cfg["project"]["name"]}')
print(f'Classes: {cfg["dataset"]["class_names"]}')


## Step 1: Verifikasi Dataset Kaggle RFT

In [ ]:
import subprocess
import os

os.makedirs('../logs', exist_ok=True)
os.makedirs('../results/visualizations', exist_ok=True)

# Verifikasi dataset yang sudah ada
from pathlib import Path
dataset_dir = Path('../data/datasets')
print(f'Dataset dir exists: {dataset_dir.exists()}')
for split in ['train', 'valid', 'test']:
    n_imgs = len(list((dataset_dir / split / 'images').glob('*'))) if (dataset_dir / split / 'images').exists() else 0
    n_lbls = len(list((dataset_dir / split / 'labels').glob('*.txt'))) if (dataset_dir / split / 'labels').exists() else 0
    print(f'  {split:6s}: {n_imgs} images, {n_lbls} labels')


## Step 2: Load and Analyze Annotations

In [ ]:
ann_path = Path('../data/raw/annotations.json')

if not ann_path.exists():
    print('⚠️  Annotations not found. Run download first.')
else:
    with open(ann_path) as f:
        taco_data = json.load(f)
    
    print(f'📊 TACO Dataset Statistics:')
    print(f'   Images     : {len(taco_data["images"])}')
    print(f'   Annotations: {len(taco_data["annotations"])}')
    print(f'   Categories : {len(taco_data["categories"])}')
    print()
    print('Categories:')
    for cat in taco_data['categories'][:15]:
        print(f'   [{cat["id"]:3d}] {cat["name"]}')


## Step 2: Analisis Distribusi Kelas

In [ ]:
# Kelas dataset Kaggle RFT (8 kelas)
CLASS_NAMES = ['bottle', 'grass', 'branch', 'milk-box', 'plastic-bag', 'plastic-garbage', 'ball', 'leaf']
CLASS_COLORS = ['#ff8000', '#00c864', '#8b5a2b', '#0078ff', '#dc0096', '#8c00ff', '#00dcdc', '#b4c83c']

# Hitung distribusi label dari dataset
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

class_counts = Counter()
for split in ['train', 'valid', 'test']:
    lbl_dir = Path('../data/datasets') / split / 'labels'
    if lbl_dir.exists():
        for lbl_file in lbl_dir.glob('*.txt'):
            for line in lbl_file.read_text().strip().splitlines():
                parts = line.split()
                if parts:
                    class_counts[int(parts[0])] += 1

print('📊 Distribusi Kelas Kaggle RFT:')
print(f'{"Kelas":<20} {"Jumlah":<10} {"Persen"}')
print('-' * 40)
total = sum(class_counts.values())
for cls_id, name in enumerate(CLASS_NAMES):
    count = class_counts.get(cls_id, 0)
    print(f'{name:<20} {count:<10} {count/total*100:.1f}%')
print(f'{"TOTAL":<20} {total}')


In [ ]:
# Plot distribusi kelas
names = CLASS_NAMES
counts = [class_counts.get(i, 0) for i in range(len(CLASS_NAMES))]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(names[::-1], counts[::-1], color=CLASS_COLORS[::-1])
ax.set_title('Kaggle RFT Dataset — Distribusi Kelas (8 Kelas)', fontsize=14, fontweight='bold')
ax.set_xlabel('Jumlah Anotasi')
for bar, count in zip(bars, counts[::-1]):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            str(count), va='center')
plt.tight_layout()
plt.savefig('../results/visualizations/class_distribution_rft.png', dpi=150)
plt.show()


## Step 3: Analisis Ukuran Objek

In [ ]:
# Analisis ukuran bounding box dari dataset
areas = []
for split in ['train', 'valid', 'test']:
    lbl_dir = Path('../data/datasets') / split / 'labels'
    img_dir = Path('../data/datasets') / split / 'images'
    if not lbl_dir.exists(): continue
    for lbl_file in lbl_dir.glob('*.txt'):
        img_files = list(img_dir.glob(lbl_file.stem + '.*'))
        if not img_files: continue
        import cv2
        img = cv2.imread(str(img_files[0]))
        if img is None: continue
        H, W = img.shape[:2]
        for line in lbl_file.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) == 5:
                _, xc, yc, w, h = int(parts[0]), *map(float, parts[1:])
                areas.append(w * W * h * H)

areas = np.array(areas)
small_mask  = areas < 1024
medium_mask = (areas >= 1024) & (areas < 9216)
large_mask  = areas >= 9216

print('📐 Distribusi Ukuran Objek:')
print(f'   Total anotasi : {len(areas)}')
print(f'   Small  (<32×32): {small_mask.sum()} ({small_mask.mean()*100:.1f}%)')
print(f'   Medium (32-96) : {medium_mask.sum()} ({medium_mask.mean()*100:.1f}%)')
print(f'   Large  (>96×96): {large_mask.sum()} ({large_mask.mean()*100:.1f}%)')
print(f'   Median area    : {np.median(areas):.0f} px²')


## Step 4: Verifikasi Struktur YOLO Dataset

In [ ]:
# Verifikasi data.yaml jika ada
import yaml
data_yaml_path = Path('../data/datasets/data.yaml')
if data_yaml_path.exists():
    with open(data_yaml_path) as f:
        data_cfg = yaml.safe_load(f)
    print('data.yaml:')
    print(f'  Classes: {data_cfg.get("nc", "?")}')
    print(f'  Names  : {data_cfg.get("names", [])}')
else:
    print('data.yaml tidak ditemukan — menggunakan kelas default:')
    print(f'  8 kelas: {CLASS_NAMES}')


## Step 5: Visualisasi Sample Gambar

In [ ]:
# Visualize a few training samples with annotations
import cv2

train_img_dir = Path('../data/datasets/train/images')
train_lbl_dir = Path('../data/datasets/train/labels')

sample_images = list(train_img_dir.glob('*'))[:6]
class_names = CLASS_NAMES

if sample_images:
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    
    for ax, img_path in zip(axes.flatten(), sample_images):
        img = cv2.imread(str(img_path))
        if img is None: continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        H, W = img.shape[:2]
        
        lbl_path = train_lbl_dir / (img_path.stem + '.txt')
        if lbl_path.exists():
            for line in lbl_path.read_text().strip().splitlines():
                parts = line.split()
                if len(parts) == 5:
                    c, xc, yc, w, h = int(parts[0]), *map(float, parts[1:])
                    x1 = int((xc - w/2) * W); y1 = int((yc - h/2) * H)
                    x2 = int((xc + w/2) * W); y2 = int((yc + h/2) * H)
                    cv2.rectangle(img_rgb, (x1,y1), (x2,y2), (255,100,0), 2)
                    cv2.putText(img_rgb, class_names[c], (x1, y1-5),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,100,0), 1)
        
        ax.imshow(img_rgb)
        ax.set_title(img_path.stem[:25], fontsize=9)
        ax.axis('off')
    
    plt.suptitle('Sample Training Images — Kaggle RFT Dataset', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../results/visualizations/sample_annotations_rft.png', dpi=150)
    plt.show()
else:
    print('No training images found.')


---

✅ **Dataset exploration complete!**

Next step: Buka `02_training.ipynb` untuk melihat konfigurasi training dan validasi model yang sudah ada.